## Regression Track — Preprocessing 


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 60)


In [2]:
train_df = pd.read_csv("data/raw/UNSW_NB15_training-set.csv")
test_df = pd.read_csv("data/raw/UNSW_NB15_testing-set.csv")

print("Training set shape:", train_df.shape)
print("Testing  set shape:", test_df.shape)
print()
print("Columns ({} total):".format(train_df.shape[1]))
print(list(train_df.columns))


Training set shape: (82332, 45)
Testing  set shape: (175341, 45)

Columns (45 total):
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']


In [3]:
train_df.dtypes.to_frame(name="dtype").T


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
dtype,int64,float64,str,str,str,int64,int64,int64,int64,float64,int64,int64,float64,float64,int64,int64,float64,float64,float64,float64,int64,int64,int64,int64,float64,float64,float64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,str,int64


In [ ]:
missing_counts = train_df.isnull().sum()
print("Columns with NaN/null values:")
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "None found.")

print()
print(f"Rows with service == '-' -> train: {(train_df['service']=='-').sum()}, "
      f"test: {(test_df['service']=='-').sum()}")


Columns with NaN/null values:
None found.

Rows with service == '-' -> train: 47153, test: 94168


In [ ]:
print("dbytes — descriptive statistics (training set):")
print(train_df["dbytes"].describe())
print()
print("Skewness of dbytes:", train_df["dbytes"].skew())
print("Skewness of log1p(dbytes):", np.log1p(train_df["dbytes"]).skew())
print()
print("Rows where dbytes == 0:", (train_df["dbytes"] == 0).sum(),
      f"({(train_df['dbytes']==0).mean()*100:.1f}% of training rows)")


dbytes — descriptive statistics (training set):
count    8.233200e+04
mean     1.323379e+04
std      1.514715e+05
min      0.000000e+00
25%      0.000000e+00
50%      1.780000e+02
75%      9.560000e+02
max      1.465753e+07
Name: dbytes, dtype: float64

Skewness of dbytes: 52.550386921780415


Skewness of log1p(dbytes): 0.2489424492101253

Rows where dbytes == 0: 36006 (43.7% of training rows)


**Observation (A1):** The training set has **82,332 rows / 45 columns**; no standard NaN values exist, but `service` uses the `'-'` placeholder for ~57% of rows (same finding as the classification-track audit; handled the same way in Section B). The chosen regression target `dbytes` is **extremely right-skewed** (raw skewness in the hundreds) with a large spike at zero (many flows never sent any response bytes, e.g. one-way scans). Taking `log1p(dbytes)` sharply reduces the skew, which is why we adopt it as the actual training target rather than raw `dbytes`.


## Section B — Preprocessing & Feature Engineering

### B1. Data Cleaning

> Note (from the audit): the raw `dbytes` target is dominated by a huge spike near zero with a long thin tail (up to ~1.47e7 bytes), which would let a handful of extreme flows dominate raw-scale regression. `log1p(dbytes)` is the training target for this reason.


In [ ]:
for df_ in (train_df, test_df):
    df_["service"] = df_["service"].replace("-", "no_service")

# --- (2) Duplicate rows ---
dup_train = train_df.duplicated().sum()
dup_test = test_df.duplicated().sum()
print(f"Duplicate rows -> train: {dup_train}, test: {dup_test}")
train_df = train_df.drop_duplicates().reset_index(drop=True)
test_df = test_df.drop_duplicates().reset_index(drop=True)
print("Shapes after de-duplication:", train_df.shape, test_df.shape)


Duplicate rows -> train: 0, test: 0


Shapes after de-duplication: (82332, 45) (175341, 45)


In [ ]:

leak_and_scope_cols = ["dmean", "dload", "attack_cat", "label", "id"]
train_df = train_df.drop(columns=leak_and_scope_cols)
test_df = test_df.drop(columns=leak_and_scope_cols)
print("Dropped columns:", leak_and_scope_cols)
print("Remaining shape -> train:", train_df.shape, " test:", test_df.shape)


Dropped columns: ['dmean', 'dload', 'attack_cat', 'label', 'id']
Remaining shape -> train: (82332, 40)  test: (175341, 40)


In [ ]:

skewed_predictor_cols = ["dur", "sbytes", "rate", "sload", "sloss", "dloss",
                          "sinpkt", "dinpkt", "sjit", "djit"]
print("Predictor columns flagged for capping (applied after the split):")
print(skewed_predictor_cols)


Predictor columns flagged for capping (applied after the split):
['dur', 'sbytes', 'rate', 'sload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit']


**Observation (B1):** No exact duplicate rows were found. `dmean`, `dload` (algebraic leaks of the target), `attack_cat`/`label` (out-of-scope classification targets), and `id` (identifier) were dropped from the feature set. Outlier capping on the skewed **predictor** columns is **deferred to after the train/validation split** (Section B2) so the IQR bounds come from the training fold only; the target `dbytes` is never capped (its skew is handled via `log1p`).


### B2. Encoding, Splitting, Outlier Capping & Scaling

Leakage-safe order of operations:

1. **Encode** categoricals (`proto` rare-grouped, then one-hot), dummy columns fixed on the **training file only**; test aligned to those columns.
2. **Split** a train/validation set (80/20) out of the training file, stratified on quantile-binned `log1p(dbytes)`; the provided test file stays as the untouched final hold-out.
3. **Cap outliers** on the skewed predictor columns using IQR bounds from **`X_train` only**, applied (transform-only) to validation and test.
4. **Engineer features** (Section B3) from the *raw, unscaled* values.
5. **Scale** all continuous features (including engineered ones) with a `StandardScaler` **fit on `X_train` only**.


In [13]:
# --- Categorical encoding ---
proto_freq = train_df["proto"].value_counts(normalize=True)
common_protos = proto_freq[proto_freq >= 0.01].index.tolist()
for df_ in (train_df, test_df):
    df_["proto"] = df_["proto"].where(df_["proto"].isin(common_protos), "other")

categorical_cols = ["proto", "service", "state"]
print({c: train_df[c].nunique() for c in categorical_cols})


{'proto': 5, 'service': 13, 'state': 7}


In [ ]:
TARGET_COL = "dbytes"

y_full = train_df[TARGET_COL].copy()
y_full_log = np.log1p(y_full)                
y_test_full = test_df[TARGET_COL].copy()
y_test_full_log = np.log1p(y_test_full)

X_full = train_df.drop(columns=[TARGET_COL])
X_test_full = test_df.drop(columns=[TARGET_COL])

X_full_enc = pd.get_dummies(X_full, columns=categorical_cols, drop_first=False)
X_test_enc = pd.get_dummies(X_test_full, columns=categorical_cols, drop_first=False)
X_test_enc = X_test_enc.reindex(columns=X_full_enc.columns, fill_value=0)   # align to train columns

print("Encoded feature matrix shape (train):", X_full_enc.shape)
print("Encoded feature matrix shape (test): ", X_test_enc.shape)


Encoded feature matrix shape (train): (82332, 61)
Encoded feature matrix shape (test):  (175341, 61)


In [ ]:

target_bins = pd.qcut(y_full_log, q=10, labels=False, duplicates="drop")

X_train, X_val, y_train_log, y_val_log, y_train, y_val = train_test_split(
    X_full_enc, y_full_log, y_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=target_bins
)

print("Train:", X_train.shape, " Val:", X_val.shape)


Train: (65865, 61)  Val: (16467, 61)


In [16]:
# --- Step 3: Outlier capping — bounds from X_train ONLY ---
cap_cols = [c for c in skewed_predictor_cols if c in X_train.columns]

outlier_bounds = {}
for col in cap_cols:
    q1, q3 = X_train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

for col, (lower, upper) in outlier_bounds.items():
    X_train[col]   = X_train[col].clip(lower, upper)
    X_val[col]     = X_val[col].clip(lower, upper)
    X_test_enc[col] = X_test_enc[col].clip(lower, upper)

print(f"Capped {len(cap_cols)} predictor columns using IQR bounds from X_train only.")


Capped 10 predictor columns using IQR bounds from X_train only.


In [ ]:
# --- Step 4: Feature engineering on RAW (unscaled) values ---
# Built BEFORE scaling so log/sum/flag features use real values and stay
# meaningful; they are scaled together with everything else in Step 5.
def engineer_features(df):
    df = df.copy()

    df["log_dur"] = np.log1p(df["dur"].clip(lower=0))

    
    df["src_activity_score"] = (df["ct_srv_src"] + df["ct_src_ltm"] +
                                 df["ct_src_dport_ltm"])

    df["is_tcp"] = df["proto_tcp"] if "proto_tcp" in df.columns else 0

    return df

X_train = engineer_features(X_train)
X_val   = engineer_features(X_val)
X_test_enc = engineer_features(X_test_enc)

print("Engineered (on raw values):", ["log_dur", "src_activity_score", "is_tcp"])
print("Shapes -> train:", X_train.shape, " val:", X_val.shape, " test:", X_test_enc.shape)


Engineered (on raw values): ['log_dur', 'src_activity_score', 'is_tcp']
Shapes -> train: (65865, 64)  val: (16467, 64)  test: (175341, 64)


In [18]:
# --- Step 5: Scaling — StandardScaler fit on X_train ONLY (engineered cols included) ---
numeric_feature_cols = [c for c in X_train.columns
                         if X_train[c].dtype != "bool" and X_train[c].nunique() > 2]

scaler = StandardScaler()
scaler.fit(X_train[numeric_feature_cols])          # fit on train ONLY

X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test_enc.copy()

X_train_scaled[numeric_feature_cols] = scaler.transform(X_train[numeric_feature_cols])
X_val_scaled[numeric_feature_cols]   = scaler.transform(X_val[numeric_feature_cols])
X_test_scaled[numeric_feature_cols]  = scaler.transform(X_test_enc[numeric_feature_cols])

print(f"Scaled {len(numeric_feature_cols)} continuous columns (incl. the 3 engineered ones).")
print("Scaler fit on X_train ONLY; validation/test transform-only.")


Scaled 37 continuous columns (incl. the 3 engineered ones).
Scaler fit on X_train ONLY; validation/test transform-only.


**Observation (B2):** Rare `proto` categories were grouped into `"other"` for generalisation. The pipeline now runs **encode → split → cap → engineer → scale**: the split precedes every fitted transform, IQR caps and the `StandardScaler` are fit on `X_train` only, and the engineered features are built from raw values then scaled alongside the rest. The split is stratified on quantile buckets of `log1p(dbytes)` so both folds see a comparable spread of small/large-response flows.


In [ ]:
X_train[["log_dur", "src_activity_score", "is_tcp"]].describe()


,log_dur,src_activity_score
count,65865.000000,65865.000000
mean,0.274384,20.938875
std,0.355075,26.728624
min,0.000000,3.000000
25%,0.000008,5.000000
50%,0.014531,10.000000
75%,0.542469,22.000000
max,1.029837,178.000000


**Observation (B3):** Because `log_dur` is now computed on the *raw* `dur` (not the scaled z-score), it spans a genuine positive range instead of collapsing most rows to zero, so it carries real information. `src_activity_score` and `is_tcp` are likewise built from raw values, then all three are scaled with the train-only scaler in Step 5 — no leakage, and every feature ends up on a comparable scale for the distance/gradient-based models (SVR, KNN, linear family).


## Save Preprocessed Data for the Regression Notebook

We persist the cleaned, encoded, scaled, feature-engineered splits — including both the log-transformed training target and the original-scale `dbytes` (for reporting RMSE/MAE back in real byte units) — so the modelling notebook (`regression.ipynb`) can load them directly.


In [20]:
import os
os.makedirs("processed_data", exist_ok=True)

X_train_scaled.assign(dbytes_log=y_train_log.values, dbytes=y_train.values)\
    .to_csv("processed_data/reg_train_processed.csv", index=False)

X_val_scaled.assign(dbytes_log=y_val_log.values, dbytes=y_val.values)\
    .to_csv("processed_data/reg_val_processed.csv", index=False)

X_test_scaled.assign(dbytes_log=y_test_full_log.values, dbytes=y_test_full.values)\
    .to_csv("processed_data/reg_test_processed.csv", index=False)

print("Saved: processed_data/reg_train_processed.csv, reg_val_processed.csv, reg_test_processed.csv")
print("Train models against the 'dbytes_log' column; convert predictions back with "
      "np.expm1(pred) and compare against the 'dbytes' column for real-unit RMSE/MAE.")


Saved: processed_data/reg_train_processed.csv, reg_val_processed.csv, reg_test_processed.csv
Train models against the 'dbytes_log' column; convert predictions back with np.expm1(pred) and compare against the 'dbytes' column for real-unit RMSE/MAE.
